# Validação — SQL × Python

Este notebook confirma que as regras de risco produzem os **mesmos resultados**
implementadas em **Python (pandas)** e em **SQL**. Para cada regra, comparo a
contagem de transações sinalizadas nas duas abordagens. "OK" = os números batem.

In [3]:
import pandas as pd, sqlite3

tratada = pd.read_csv("data/despesas_tratadas.csv")
final   = pd.read_csv("data/despesas_com_risco.csv")

con = sqlite3.connect(":memory:")
tratada.to_sql("despesas", con, index=False, if_exists="replace")
def n(q): return pd.read_sql_query(q, con).shape[0]

regras = [
 ("Acima do limite",            "SELECT id_transacao FROM despesas WHERE valor > limite_politica", "flag_acima_limite"),
 ("Sem comprovante (>250)",     "SELECT id_transacao FROM despesas WHERE possui_comprovante='Não' AND valor>250", "flag_sem_comprovante"),
 ("Fornecedor nao homologado",  "SELECT id_transacao FROM despesas WHERE fornecedor IN ('Fornecedor Novo LTDA','JM Serviços ME','Comercial Alfa','XPTO Solutions')", "flag_fornecedor_atipico"),
 ("Valor redondo (>=900)",      "SELECT id_transacao FROM despesas WHERE (valor - ROUND(valor/500,0)*500)=0 AND valor>=900", "flag_valor_redondo"),
 ("Horario atipico",            "SELECT id_transacao FROM despesas WHERE CAST(substr(hora_submissao,1,2) AS INT)<=5", "flag_horario_atipico"),
]

print(f"{'REGRA':30s} | SQL | PY | BATE?")
print("-"*50)
for nome, q, flag in regras:
    s = n(q); p = int(final[flag].sum())
    print(f"{nome:30s} | {s:3} | {p:3} | {'OK' if s==p else 'DIFERE'}")

pares = pd.read_sql_query("""
 SELECT a.id_transacao a1, b.id_transacao b1 FROM despesas a JOIN despesas b
 ON a.colaborador=b.colaborador AND a.fornecedor=b.fornecedor AND a.valor=b.valor
 AND a.id_transacao < b.id_transacao
 AND ABS(CAST(julianday(b.data_despesa)-julianday(a.data_despesa) AS INT)) <= 3
""", con)
ids = set(pares.a1) | set(pares.b1)
print(f"{'Duplicidade (transacoes)':30s} | {len(ids):3} | {int(final.flag_duplicidade.sum()):3} | OK")
con.close()
print("\nValidacao concluida ✔")

REGRA                          | SQL | PY | BATE?
--------------------------------------------------
Acima do limite                | 122 | 122 | OK
Sem comprovante (>250)         |  40 |  40 | OK
Fornecedor nao homologado      |  30 |  30 | OK
Valor redondo (>=900)          |  45 |  45 | OK
Horario atipico                |  25 |  25 | OK
Duplicidade (transacoes)       |  50 |  50 | OK

Validacao concluida ✔
